# Qwen2 news value classification and calibration

This notebook operationalises six news values for NU.nl articles that mention Dutch locations: entertainment, bad news, magnitude, good news, celebrity, and power elite. Each article receives a score of `0`, `50`, or `100` for each value.

The notebook starts from the article and location files created in Notebook 1:

- `articles_final.csv`
- `locations_final.csv`

The main article-level outputs are:

- `article_news_values.csv` — raw Qwen2 classifications;
- `article_news_values_raw_outputs.csv` — unparsed model responses for auditability;
- `article_news_values_calibrated_to_harcup_oneill.csv` — benchmark-calibrated scores used for the main GIS workflow;
- `manual_validation_sample.csv` — sample used for manual validation.

The full Qwen2 classification was completed in an earlier run. In this submitted version, `RUN_QWEN_MODEL = False`, so the notebook reuses the saved classification files rather than rerunning the expensive model step. I used ChatGPT as coding support for selected implementation tasks, especially prompt construction, JSON parsing, calibration logic, and validation preparation. The scoring scheme, methodological choices, checks, and interpretation remain part of my thesis workflow.

In [ ]:
# Install runtime dependencies when using a fresh Colab environment.
!pip install -q --upgrade-strategy only-if-needed \
  "numpy>=1.26,<2.1" \
  "pandas==2.2.2" \
  "protobuf>=5.29.1,<6" \
  "transformers>=4.45,<5" \
  "accelerate>=0.33" \
  "bitsandbytes" \
  "sentencepiece" \
  "einops" \
  "tqdm"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.5 MB/s eta 0:00:00


In [ ]:
# Import libraries
import gc
import json
import re
import time
from pathlib import Path
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Report GPU availability for reproducibility of optional Qwen2 inference.
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU available. This is fine if the model classification is not being rerun.")

Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# Mount Google Drive so the notebook can read and write the thesis CSV files.
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2. Configuration

This section defines all input, output, diagnostic, and model settings used in the notebook. The input paths match the final files produced by Notebook 1.

`RUN_QWEN_MODEL` is set to `False` in the submitted version. This means that the notebook loads the saved classification outputs instead of generating new model predictions. To reproduce the original full inference run, the saved few-shot file and the same Qwen2 configuration should be kept with the project files.



In [ ]:
# Define all project paths centrally to keep the three-notebook workflow consistent.
PROJECT_DIR = Path("/content/drive/MyDrive/Thesis")
OUTPUT_DIR = PROJECT_DIR / "data_processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Input files from the previous notebook
ARTICLES_PATH = OUTPUT_DIR / "articles_final.csv"
LOCATIONS_PATH = OUTPUT_DIR / "locations_final.csv"

# Raw Qwen2 classification outputs
NEWS_VALUES_PATH = OUTPUT_DIR / "article_news_values.csv"
RAW_OUTPUTS_PATH = OUTPUT_DIR / "article_news_values_raw_outputs.csv"

# Calibrated classification outputs
CALIBRATED_NEWS_VALUES_PATH = OUTPUT_DIR / "article_news_values_calibrated_to_harcup_oneill.csv"
CALIBRATION_SUMMARY_PATH = OUTPUT_DIR / "news_value_calibration_summary.csv"

# GIS outputs based on calibrated news values
CALIBRATED_FINAL_GIS_PATH = OUTPUT_DIR / "location_mentions_with_calibrated_news_values_for_gis.csv"
CALIBRATED_SUMMARY_BY_LOCATION_PATH = OUTPUT_DIR / "location_news_value_summary_by_location_calibrated.csv"

# Pilot, few-shot, and validation files
PILOT_PATH = OUTPUT_DIR / "pilot_article_news_values.csv"
PILOT_CALIBRATED_PATH = OUTPUT_DIR / "pilot_article_news_values_calibrated_to_harcup_oneill.csv"
PILOT_CALIBRATION_SUMMARY_PATH = OUTPUT_DIR / "pilot_news_value_calibration_summary.csv"

VALIDATION_SAMPLE_PATH = OUTPUT_DIR / "manual_validation_sample.csv"
VALIDATION_METRICS_PATH = OUTPUT_DIR / "validation_metrics.csv"
FEWSHOT_EXAMPLES_PATH = OUTPUT_DIR / "manual_fewshot_examples.csv"

# Diagnostic files
ARTICLE_ID_MAPPING_PATH = OUTPUT_DIR / "article_id_mapping.csv"
DUPLICATE_ARTICLES_PATH = OUTPUT_DIR / "diagnostic_duplicate_article_id_rows.csv"
DUPLICATE_LOCATION_ROWS_PATH = OUTPUT_DIR / "diagnostic_duplicate_article_location_rows.csv"
EXCLUDED_NON_NL_LOCATIONS_PATH = OUTPUT_DIR / "diagnostic_excluded_non_netherlands_location_rows.csv"

# General settings
BODY_CHAR_LIMIT = 750
CLASSIFY_ONLY_ARTICLES_WITH_LOCATIONS = True
SAVE_EVERY_N_ARTICLES = 25
RANDOM_SEED = 42
MAX_ARTICLES_TO_CLASSIFY = None

# Sample sizes
PILOT_N = 150
FEWSHOT_SAMPLE_N = 40
FEWSHOT_EXAMPLES_IN_PROMPT = 4
FEWSHOT_EXAMPLE_CHAR_LIMIT = 900
VALIDATION_N = 300

# Qwen2 model settings
MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"
MAX_CLASSIFICATION_RETRIES = 3

# Set to True only when deliberately reproducing or replacing the model inference step.
RUN_QWEN_MODEL = False

# Article column names
ARTICLE_ID_COL = "article_id"
TITLE_COL = "title"
DATE_COL = "date"
SUBJECTS_COL = "subjects"
LEAD_COL = "lead_text"
BODY_COL = "body_text"
ARTICLE_LOCATIONS_COL = "locations"

# Location column names
LOCATION_ARTICLE_ID_COL = "article_id"
LOCATION_NAME_COL = "location"
LAT_COL = "lat"
LON_COL = "lon"

# News value score columns
NEWS_VALUES = [
    "entertainment_score",
    "bad_news_score",
    "magnitude_score",
    "good_news_score",
    "celebrity_score",
    "power_elite_score",
]

# Print file-availability checks for the saved workflow inputs and outputs.
print("Project directory:", PROJECT_DIR)
print("Articles file exists:", ARTICLES_PATH.exists())
print("Locations file exists:", LOCATIONS_PATH.exists())
print("Run Qwen model:", RUN_QWEN_MODEL)
print("Raw Qwen2 output exists:", NEWS_VALUES_PATH.exists())
print("Calibrated output exists:", CALIBRATED_NEWS_VALUES_PATH.exists())

Project directory: /content/drive/MyDrive/Thesis
Articles file exists: True
Locations file exists: True
Run Qwen model: False
Raw Qwen2 output exists: True
Calibrated output exists: True


## 3. Load and prepare input data

This section loads the article and location files produced by Notebook 1. It checks the required columns and prepares stable article identifiers so that article-level Qwen2 scores can later be linked back to article-location mentions.


In [ ]:
# Fail early if an expected column is missing from an input table.
def require_columns(df, required, frame_name):
    missing = []

    for col in required:
        if col not in df.columns:
            missing.append(col)

    if missing:
        raise ValueError(frame_name + " is missing required columns: " + str(missing))

# Keep article_id as the first column in saved outputs.
def move_column_to_front(df, column):
    other_columns = [col for col in df.columns if col != column]
    return df[[column] + other_columns].copy()

# Load the article and location CSV files from Google Drive.
def load_input_data():
    if not ARTICLES_PATH.exists():
        raise FileNotFoundError("Missing articles file: " + str(ARTICLES_PATH))

    if not LOCATIONS_PATH.exists():
        raise FileNotFoundError("Missing locations file: " + str(LOCATIONS_PATH))

    articles = pd.read_csv(ARTICLES_PATH)
    locations = pd.read_csv(LOCATIONS_PATH)

    # Remove duplicated columns that can appear after repeated CSV/export steps.
    articles = articles.loc[:, ~articles.columns.duplicated()].copy()
    locations = locations.loc[:, ~locations.columns.duplicated()].copy()

    require_columns(
        articles,
        [ARTICLE_ID_COL, TITLE_COL, DATE_COL, SUBJECTS_COL, LEAD_COL, BODY_COL, ARTICLE_LOCATIONS_COL],
        "articles_final.csv",
    )

    require_columns(
        locations,
        [LOCATION_ARTICLE_ID_COL, LOCATION_NAME_COL, LAT_COL, LON_COL],
        "locations_final.csv",
    )

    return articles, locations


# Load the input tables and display a small structural check.
articles_df, locations_df = load_input_data()

print("Loaded article rows:", len(articles_df))
print("Loaded location rows:", len(locations_df))

display(articles_df.head(3))
display(locations_df.head(3))

Loaded article rows: 55772
Loaded location rows: 19802


,article_id,file,file_name,title,date,subjects,subjects_json,body_text,lead_text,body_char_count,...,locations,locations_json,location_count,has_dutch_location,entertainment_score,bad_news_score,magnitude_score,good_news_score,celebrity_score,power_elite_score
0,'Vrienden' Poetin en Xi verstevigen band in vi...,/content/drive/MyDrive/Thesis/data/01-2025/'Vr...,'Vrienden' Poetin en Xi verstevigen band in vi...,'Vrienden' Poetin en Xi verstevigen band in vi...,21 januari 2025 dinsdag 10:11 PM GMT,Heads Of State + Government,"[{""subject"": ""Heads Of State + Government"", ""p...",De Russische president Vladimir Poetin heeft d...,De Russische president Vladimir Poetin heeft d...,188,...,NaN,[],0,False,NaN,NaN,NaN,NaN,NaN,NaN
1,10 procent van huishoudens bezit ruim de helft...,/content/drive/MyDrive/Thesis/data/01-2025/10 ...,10 procent van huishoudens bezit ruim de helft...,10 procent van huishoudens bezit ruim de helft...,14 januari 2025 dinsdag 03:41 PM GMT,Housing Market; Population + Demographics; Tax...,"[{""subject"": ""Housing Market"", ""percentage"": 8...",56 procent van het Nederlandse vermogen is in ...,56 procent van het Nederlandse vermogen is in ...,2401,...,NaN,[],0,False,NaN,NaN,NaN,NaN,NaN,NaN
2,10 procent van huishoudens bezit ruim helft va...,/content/drive/MyDrive/Thesis/data/01-2025/10 ...,10 procent van huishoudens bezit ruim helft va...,10 procent van huishoudens bezit ruim helft va...,14 januari 2025 dinsdag 03:41 PM GMT,Housing Market; Population + Demographics; Tax...,"[{""subject"": ""Housing Market"", ""percentage"": 9...",Het Nederlandse vermogen is voor 56 procent in...,Het Nederlandse vermogen is voor 56 procent in...,2136,...,NaN,[],0,False,NaN,NaN,NaN,NaN,NaN,NaN


,article_id,file,file_name,title,date,subjects,location,lat,lon
0,12 mensen (erg) ziek_ dit weten we over het he...,/content/drive/MyDrive/Thesis/data/01-2025/12 ...,12 mensen (erg) ziek_ dit weten we over het he...,12 mensen (erg) ziek: dit weten we over het he...,14 januari 2025 dinsdag 09:52 AM GMT,Government Departments + Authorities; Hepatiti...,Wageningen,51.966302,5.666281
1,"12 zieken, 2 opnames_ dit weten we over het he...",/content/drive/MyDrive/Thesis/data/01-2025/12 ...,"12 zieken, 2 opnames_ dit weten we over het he...","12 zieken, 2 opnames: dit weten we over het he...",14 januari 2025 dinsdag 09:52 AM GMT,Hepatitis; Infectious Disease; Viruses; Govern...,Wageningen,51.966302,5.666281
2,125.000 mensen bezochten nepwebshop waarmee po...,/content/drive/MyDrive/Thesis/data/01-2025/125...,125.000 mensen bezochten nepwebshop waarmee po...,125.000 mensen bezochten nepwebshop waarmee po...,25 januari 2025 zaterdag 09:06 AM GMT,Fraud + Financial Crime; Mail Order + Internet...,Heemskerk,52.510433,4.672354


### Create numeric article IDs

The article IDs produced in Notebook 1 are long text-based identifiers. For the classification and GIS stages, I create shorter numeric article IDs while preserving the original IDs in `article_id_mapping.csv`.

This mapping connects four levels of the workflow:

1. the article table;
2. the location table;
3. the Qwen2 classification results;
4. the GIS-ready article-location output.

I used ChatGPT as coding support to check the join logic for this data-linking step.

In [ ]:
def create_numeric_article_ids(articles, locations):
    # Assign stable numeric article IDs while preserving the original article identifiers.
    articles = articles.rename(columns={ARTICLE_ID_COL: "article_id_original"}).copy()
    locations = locations.rename(columns={LOCATION_ARTICLE_ID_COL: "article_id_original"}).copy()

    # Use shared article metadata to keep the article and location tables aligned.
    key_candidates = ["article_id_original", "file_name", TITLE_COL, DATE_COL]
    key_cols = [col for col in key_candidates if col in articles.columns and col in locations.columns]
    if "article_id_original" not in key_cols:
        raise ValueError("Both input files must contain the original article_id column.")

    # Build and save a mapping table so numeric IDs remain traceable to the original records.
    article_id_map = articles[key_cols].drop_duplicates().copy()
    article_id_map["_date_sort"] = (
        pd.to_datetime(article_id_map[DATE_COL], errors="coerce")
        if DATE_COL in article_id_map.columns
        else pd.NaT
    )
    sort_cols = ["_date_sort"] + [col for col in ["file_name", TITLE_COL, "article_id_original"] if col in article_id_map.columns]
    article_id_map = article_id_map.sort_values(sort_cols, na_position="last").reset_index(drop=True)
    article_id_map.insert(0, "article_id", range(1, len(article_id_map) + 1))
    article_id_map = article_id_map.drop(columns="_date_sort")

    # Add the numeric article ID back to both dataframes.
    articles = articles.merge(article_id_map, on=key_cols, how="left")
    locations = locations.merge(article_id_map, on=key_cols, how="left")

    # Verify that every article and location row receives a numeric ID before continuing.
    if articles["article_id"].isna().any() or locations["article_id"].isna().any():
        raise ValueError("At least one row could not be matched to a numeric article_id.")

    articles = articles.drop(columns="article_id_original")
    locations = locations.drop(columns="article_id_original")
    articles = move_column_to_front(articles, "article_id")
    locations = move_column_to_front(locations, "article_id")

    # Save the mapping file so the numeric IDs can be traced back later.
    article_id_map.to_csv(ARTICLE_ID_MAPPING_PATH, index=False, encoding="utf-8-sig")
    return articles, locations, article_id_map


articles_df, locations_df, article_id_map = create_numeric_article_ids(articles_df, locations_df)

print("Numeric article IDs:", article_id_map["article_id"].nunique())
print("Mapping saved to:", ARTICLE_ID_MAPPING_PATH)
display(article_id_map.head())

/tmp/ipykernel_1775/3854958132.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(article_id_map[DATE_COL], errors="coerce")


Numeric article IDs: 55772
Mapping saved to: /content/drive/MyDrive/Thesis/data_processed/article_id_mapping.csv


,article_id,article_id_original,file_name,title,date
0,1,' Amazon werkt aan relatief dure tablet met An...,' Amazon werkt aan relatief dure tablet met An...,'Amazon werkt aan relatief dure tablet met And...,21 augustus 2025 donderdag 12:04 PM GMT
1,2,' Apple overweegt Google -spraakassistent te g...,' Apple overweegt Google -spraakassistent te g...,'Apple overweegt Google-spraakassistent te geb...,23 augustus 2025 zaterdag 09:56 AM GMT
2,3,"' Dubai ', 'oviebol' of naturel_ welke oliebol...","' Dubai ', 'oviebol' of naturel_ welke oliebol...","'Dubai', 'oviebol' of naturel: welke oliebol i...",31 december 2025 woensdag 10:15 AM GMT
3,4,' Ferrari is niks met Hamilton opgeschoten',' Ferrari is niks met Hamilton opgeschoten'.DOCX,'Ferrari is niks met Hamilton opgeschoten',3 augustus 2025 zondag 10:04 PM GMT
4,5,' India zet militaire acties tegen Pakistan sl...,' India zet militaire acties tegen Pakistan sl...,'India zet militaire acties tegen Pakistan sle...,12 mei 2025 maandag 09:25 PM GMT


### Preprocess articles and locations

This step standardises the article and location tables before classification. It removes duplicate article rows, keeps only coordinates within broad Netherlands bounds, removes duplicate article-location rows, and keeps only articles that still have at least one valid Dutch location.

Rows removed during preprocessing are saved as diagnostic CSV files. This makes the filtering decisions inspectable rather than silently changing the corpus.


In [ ]:
def preprocess_articles_and_locations(articles, locations):
    articles = articles.copy()
    locations = locations.copy()

    # Record row counts before filtering for the preprocessing summary.
    original_article_rows = len(articles)
    original_location_rows = len(locations)

    # Make article IDs comparable across the two tables.
    articles["article_id"] = articles["article_id"].astype(str)
    locations["article_id"] = locations["article_id"].astype(str)

    # Make sure latitude and longitude are numeric.
    locations[LAT_COL] = pd.to_numeric(locations[LAT_COL], errors="coerce")
    locations[LON_COL] = pd.to_numeric(locations[LON_COL], errors="coerce")

    # Remove duplicate article rows.
    duplicate_article_rows = articles[
        articles.duplicated(subset=["article_id"], keep=False)
    ].copy()

    duplicate_article_rows.to_csv(
        DUPLICATE_ARTICLES_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    articles = articles.drop_duplicates(subset=["article_id"], keep="first").copy()

    # Keep only coordinates that are roughly inside the Netherlands.
    # These are broad bounds, mainly used to remove obvious geocoding mistakes.
    inside_netherlands = (
        locations[LAT_COL].between(50.6, 53.7)
        & locations[LON_COL].between(3.2, 7.3)
    )

    excluded_locations = locations.loc[~inside_netherlands].copy()

    excluded_locations.to_csv(
        EXCLUDED_NON_NL_LOCATIONS_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    locations = locations.loc[inside_netherlands].copy()

    # Remove duplicate article-location rows.
    location_dedup_cols = ["article_id", LOCATION_NAME_COL, LAT_COL, LON_COL]

    duplicate_location_rows = locations[
        locations.duplicated(subset=location_dedup_cols, keep=False)
    ].copy()

    duplicate_location_rows.to_csv(
        DUPLICATE_LOCATION_ROWS_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    locations = locations.drop_duplicates(
        subset=location_dedup_cols,
        keep="first",
    ).copy()

    # Keep only locations that belong to articles still present in the article table.
    # Remove location rows that no longer have a matching article row.
    valid_article_ids = set(articles["article_id"])
    locations = locations[locations["article_id"].isin(valid_article_ids)].copy()

    # Only classify articles that mention at least one valid Dutch location.
    if CLASSIFY_ONLY_ARTICLES_WITH_LOCATIONS:
        located_ids = set(locations["article_id"])
        articles_to_classify = articles[
            articles["article_id"].isin(located_ids)
        ].copy()
    else:
        articles_to_classify = articles.copy()

    # The resulting table defines the article set used for Qwen2 classification.
    # Optional testing control: limit the corpus only when deliberately debugging the notebook.
    if MAX_ARTICLES_TO_CLASSIFY is not None:
        articles_to_classify = articles_to_classify.sample(
            n=min(MAX_ARTICLES_TO_CLASSIFY, len(articles_to_classify)),
            random_state=RANDOM_SEED,
        ).copy()

    # Create a short summary so I can check what changed during preprocessing.
    summary = pd.DataFrame([
        {"step": "original_article_rows", "n": original_article_rows},
        {"step": "duplicate_article_rows", "n": len(duplicate_article_rows)},
        {"step": "article_rows_after_deduplication", "n": len(articles)},
        {"step": "original_location_rows", "n": original_location_rows},
        {"step": "excluded_non_netherlands_location_rows", "n": len(excluded_locations)},
        {"step": "duplicate_location_rows", "n": len(duplicate_location_rows)},
        {"step": "location_rows_after_filtering", "n": len(locations)},
        {"step": "articles_available_for_classification", "n": len(articles_to_classify)},
    ])

    return articles, locations, articles_to_classify, summary


articles_df, locations_df, articles_to_classify, preprocessing_summary = preprocess_articles_and_locations(
    articles_df,
    locations_df,
)

display(preprocessing_summary)
display(articles_to_classify.head(3))

,step,n
0,original_article_rows,55772
1,duplicate_article_rows,0
2,article_rows_after_deduplication,55772
3,original_location_rows,19802
4,excluded_non_netherlands_location_rows,28
5,duplicate_location_rows,0
6,location_rows_after_filtering,19774
7,articles_available_for_classification,12225


,article_id,file,file_name,title,date,subjects,subjects_json,body_text,lead_text,body_char_count,...,locations,locations_json,location_count,has_dutch_location,entertainment_score,bad_news_score,magnitude_score,good_news_score,celebrity_score,power_elite_score
11,436,/content/drive/MyDrive/Thesis/data/01-2025/12 ...,12 mensen (erg) ziek_ dit weten we over het he...,12 mensen (erg) ziek: dit weten we over het he...,14 januari 2025 dinsdag 09:52 AM GMT,Government Departments + Authorities; Hepatiti...,"[{""subject"": ""Government Departments + Authori...",Albert Heijn riep maandag alle diepvrieszakken...,Albert Heijn riep maandag alle diepvrieszakken...,6219,...,Wageningen,"[{""location"": ""Wageningen"", ""lat"": 51.9663016,...",1,True,NaN,NaN,NaN,NaN,NaN,NaN
12,438,/content/drive/MyDrive/Thesis/data/01-2025/12 ...,"12 zieken, 2 opnames_ dit weten we over het he...","12 zieken, 2 opnames: dit weten we over het he...",14 januari 2025 dinsdag 09:52 AM GMT,Hepatitis; Infectious Disease; Viruses; Govern...,"[{""subject"": ""Hepatitis"", ""percentage"": 100}, ...",Albert Heijn riep maandag alle diepvrieszakken...,Albert Heijn riep maandag alle diepvrieszakken...,5445,...,Wageningen,"[{""location"": ""Wageningen"", ""lat"": 51.9663016,...",1,True,NaN,NaN,NaN,NaN,NaN,NaN
13,443,/content/drive/MyDrive/Thesis/data/01-2025/125...,125.000 mensen bezochten nepwebshop waarmee po...,125.000 mensen bezochten nepwebshop waarmee po...,25 januari 2025 zaterdag 09:06 AM GMT,Fraud + Financial Crime; Mail Order + Internet...,"[{""subject"": ""Fraud + Financial Crime"", ""perce...",Ruim 125.000 bezoekers bezochten de afgelopen ...,Ruim 125.000 bezoekers bezochten de afgelopen ...,2235,...,Heemskerk,"[{""location"": ""Heemskerk"", ""lat"": 52.5104331, ...",1,True,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Build article excerpts for Qwen2

For each article, I construct the text field that is passed to Qwen2. The prompt context includes the title, date, subjects, detected Dutch locations, and a shortened article excerpt.

The lead text is used when it contains enough information. If the lead is missing or too short, the body text is used instead.



### Create the classification text

This cell creates `classification_text`, the article-level prompt text used for Qwen2 classification.

The full article text is not always necessary for the news-value task, so the prompt keeps the most relevant context: title, date, subjects, detected Dutch locations, and a controlled excerpt. `BODY_CHAR_LIMIT` keeps prompt length relatively consistent across articles and reduces model/runtime variability.



In [ ]:
# Normalise text fields before inserting them into prompts.
def clean_for_prompt(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


# Preserve the order of detected location names while removing duplicates.
def unique_in_order(values):
    seen = set()
    unique_values = []
    for value in values.dropna().astype(str):
        value = value.strip()
        if value and value not in seen:
            unique_values.append(value)
            seen.add(value)
    return unique_values


# Link each article ID to the locations detected in Notebook 1.
def build_location_lookup(locations):
    grouped = locations.groupby("article_id", dropna=False)[LOCATION_NAME_COL].apply(unique_in_order)
    return {str(article_id): locs for article_id, locs in grouped.items()}


article_location_lookup = build_location_lookup(locations_df)


# Format the detected locations as a short string for the prompt.
def get_locations_for_article(article_id, max_locations=20):
    locs = article_location_lookup.get(str(article_id), [])
    if len(locs) > max_locations:
        return "; ".join(locs[:max_locations]) + f"; ... ({len(locs)} total)"
    return "; ".join(locs)


# Build the final article excerpt that Qwen2 will classify.
def build_classification_text(row, body_char_limit=BODY_CHAR_LIMIT):
    title = clean_for_prompt(row.get(TITLE_COL, ""))
    date = clean_for_prompt(row.get(DATE_COL, ""))
    subjects = clean_for_prompt(row.get(SUBJECTS_COL, ""))
    article_locations = clean_for_prompt(row.get(ARTICLE_LOCATIONS_COL, ""))
    detected_locations = get_locations_for_article(row["article_id"])

    lead = clean_for_prompt(row.get(LEAD_COL, ""))
    body = clean_for_prompt(row.get(BODY_COL, ""))

    # Prefer informative lead text. Otherwise fall back to the body text.
    if lead and len(lead) >= 80:
        source_text = lead
        source_name = "lead_text"
        if len(source_text) < 600 and body:
            source_text = f"{source_text} {body}"
            source_name = "lead_text + body_text"
    else:
        source_text = body
        source_name = "body_text"

    # Truncate article text to keep prompts comparable across the corpus.
    if body_char_limit is not None:
        source_text = source_text[:body_char_limit]

    return "\n".join([
        f"Article ID: {row['article_id']}",
        f"Date: {date}",
        f"Title: {title}",
        f"Subjects: {subjects}",
        f"Detected Dutch locations: {detected_locations or article_locations}",
        f"Text source used: {source_name}",
        f"Article excerpt: {source_text}",
    ])


# Apply the prompt-building function to every article.
articles_to_classify["classification_text"] = articles_to_classify.apply(build_classification_text, axis=1)
articles_to_classify["classification_text_char_count"] = articles_to_classify["classification_text"].str.len()

display(articles_to_classify["classification_text_char_count"].describe())
print(articles_to_classify["classification_text"].iloc[0][:2000])

,classification_text_char_count
count,12225.000000
mean,1061.380613
std,174.524945
min,277.000000
25%,1030.000000
50%,1078.000000
75%,1140.000000
max,1876.000000


Article ID: 436
Date: 14 januari 2025 dinsdag 09:52 AM GMT
Title: 12 mensen (erg) ziek: dit weten we over het hepatitisvirus in AH-bessen
Subjects: Government Departments + Authorities; Hepatitis; Viruses; Health Care Regulation + Policy; Infectious Disease; Public Health; Health Care Professionals; Food Inspection; Food + Beverage Regulation + Policy; Colleges + Universities
Detected Dutch locations: Wageningen
Text source used: lead_text
Article excerpt: Albert Heijn riep maandag alle diepvrieszakken blauwe bessen van 1 kilo terug vanwege mogelijke besmetting met het hepatitis A-virus. Nu blijkt dat zeker twaalf mensen die de bessen aten (ernstig) ziek zijn geworden en het RIVM houdt rekening met nog veel meer besmettingen. "We weten nu van twaalf mensen dat ze het virus hebben. Twee van hen zijn ernstig ziek en liggen in het ziekenhuis", vertelt RIVM-woordvoerder Eelco Franz aan NU.nl. Mogelijk zijn er honderden mensen besmet met het virus. "Dat is geen reden tot zorg, want de meest

## 5. Codebook and few-shot examples

This section defines the six news values used in the Qwen2 classification. The model is instructed to assign each value a score of `0`, `50`, or `100`.

The saved full classification file was produced in an earlier run with four completed few-shot examples. In the current submitted run, Qwen2 is disabled and the notebook may show zero completed examples if the manual few-shot file is not present in the active workspace.

I used ChatGPT as coding support for the prompt-building and JSON-parsing implementation. I made the final decisions about the codebook, manual examples, and interpretation of the resulting scores.


In [ ]:
# Insert the codebook into every prompt so Qwen2 applies the same operational definitions across articles.
PROMPT_CODEBOOK = """
News values to classify:

1. entertainment_score
Soft or engaging newsworthiness: sport, culture, media, showbusiness, lifestyle, leisure, animals, festivals, humour, spectacle, or lighter human interest.

2. bad_news_score
Negative newsworthiness: harm, death, injury, illness, danger, loss, damage, crime, punishment, crisis, failure, threat, or serious negative consequences.

3. magnitude_score
Scale or impact: large numbers, many people affected, major financial amounts, broad consequences, national or regional impact, records, exceptional scale, or extreme occurrence.

4. good_news_score
Positive newsworthiness: success, victory, achievement, recovery, rescue, improvement, celebration, breakthrough, positive return, favourable outcome, or a problem being solved.

5. celebrity_score
Famous people: well-known athletes, royals, artists, presenters, influencers, entertainers, or other public figures whose public profile contributes to the story.

6. power_elite_score
Powerful actors or institutions: government, police, courts, ministries, municipalities, regulators, major companies, public authorities, hospitals, universities, or other institutions when they are central actors.

Scoring:
0 = not meaningfully present.
50 = present but secondary.
100 = central to why the article is newsworthy.

Multi-label rule:
Evaluate all six values independently. Several values may be present in the same article.

dominant_news_values:
Return the value or values with the highest non-zero score. Use ["none"] only if all six scores are 0.
""".strip()

# Only these three score values are allowed in the final output.
VALID_SCORE_VALUES = {0, 50, 100}

# Convert model scores to one of the allowed values: 0, 50, or 100.
def coerce_score(value):
    if isinstance(value, str):
        value = value.strip().replace("%", "")
        mapping = {
            "not present": 0,
            "none": 0,
            "no": 0,
            "slightly present": 50,
            "slight": 50,
            "secondary": 50,
            "strongly present": 100,
            "strong": 100,
            "central": 100,
            "yes": 100,
        }
        if value.lower() in mapping:
            return mapping[value.lower()]
    try:
        numeric_value = float(value)
    except Exception:
        return 0
    return min(VALID_SCORE_VALUES, key=lambda score: abs(score - numeric_value))


# Identify which news value or values have the highest non-zero score.
def score_dict_to_dominant_values(score_dict):
    scores = {col.replace("_score", ""): coerce_score(score_dict.get(col, 0)) for col in NEWS_VALUES}
    top_score = max(scores.values()) if scores else 0
    if top_score <= 0:
        return ["none"]
    return [label for label, score in scores.items() if score == top_score]


# Convert the list of dominant values into a readable string for the CSV file.
def dominant_values_to_string(values):
    return "; ".join(values) if values else "none"

### Create manual few-shot examples
The notebook first creates a template file. I then manually fill in the six news-value scores for selected articles. Only rows with complete manual scores are used as few-shot examples. If the completed few-shot file already exists, the notebook loads it instead of creating a new one.


## 6. Load the language model

This step is only needed when I want to run new Qwen2 classifications. Since the full classification has already been saved, model loading is switched off by default.

If `RUN_QWEN_MODEL` is `False`, the notebook uses the existing CSV files instead of loading Qwen2 again.


In [ ]:
# Create a template for manually scored few-shot examples.
def create_fewshot_template():
    sample = articles_to_classify.sample(
        n=min(FEWSHOT_SAMPLE_N, len(articles_to_classify)),
        random_state=RANDOM_SEED,
    ).copy()

    keep_cols = ["article_id", TITLE_COL, DATE_COL, SUBJECTS_COL, "classification_text"]
    sample = sample[[col for col in keep_cols if col in sample.columns]].copy()

    # Add empty manual scoring columns for each news value.
    for col in NEWS_VALUES:
        sample[f"manual_{col}"] = ""
    sample["manual_note"] = ""

    sample.to_csv(FEWSHOT_EXAMPLES_PATH, index=False, encoding="utf-8-sig")
    print("Few-shot template saved:", FEWSHOT_EXAMPLES_PATH)


# Use only examples where all six manual score columns have been completed.
def load_completed_fewshot_examples():
    if not FEWSHOT_EXAMPLES_PATH.exists():
        create_fewshot_template()
        return pd.DataFrame()

    examples = pd.read_csv(FEWSHOT_EXAMPLES_PATH)
    manual_cols = [f"manual_{col}" for col in NEWS_VALUES]
    require_columns(examples, manual_cols, "manual_fewshot_examples.csv")

    # A row counts as completed only when every manual score column has a value.
    filled = examples[manual_cols].notna().all(axis=1)
    filled &= examples[manual_cols].astype(str).apply(lambda col: col.str.strip().ne("")).all(axis=1)
    completed = examples.loc[filled].copy()

    for col in NEWS_VALUES:
        completed[col] = completed[f"manual_{col}"].apply(coerce_score)

    if not completed.empty:
        completed["article_id"] = completed["article_id"].astype(str)
        completed = completed.drop_duplicates(subset="article_id", keep="first")

    print("Completed few-shot examples:", len(completed))
    return completed


# These completed examples can be included in the prompt later.
completed_fewshot_examples = load_completed_fewshot_examples()
fewshot_examples_for_prompt = completed_fewshot_examples.head(FEWSHOT_EXAMPLES_IN_PROMPT).copy()
print("Few-shot examples used in each prompt:", len(fewshot_examples_for_prompt))

display(fewshot_examples_for_prompt.head())

Completed few-shot examples: 0
Few-shot examples used in each prompt: 0


,article_id,title,date,subjects,classification_text,manual_entertainment_score,manual_bad_news_score,manual_magnitude_score,manual_good_news_score,manual_celebrity_score,manual_power_elite_score,manual_note,entertainment_score,bad_news_score,magnitude_score,good_news_score,celebrity_score,power_elite_score


**Note.** The saved full classification file was produced in an earlier Qwen2 run with four completed few-shot examples. In this submitted notebook, `RUN_QWEN_MODEL = False`, so the model is not rerun and the saved classification file is reused. If the active workspace does not contain the completed few-shot file, the cell above may report zero completed examples; this does not change the already saved full classification output.

## 6. Load the language model

Model loading is required only when generating new Qwen2 classifications. For the submitted workflow, the full classification output already exists and `RUN_QWEN_MODEL = False`, so the notebook skips model loading and continues with the saved CSV files.



### Load Qwen2

This cell loads `Qwen/Qwen2-1.5B-Instruct` from Hugging Face when a new inference run is required. The step depends on GPU memory and the `transformers` library.

I used ChatGPT as coding support for the model-loading implementation, while retaining the model choice and run configuration used in the thesis workflow.


In [ ]:
# Initialise model variables only when a new Qwen2 inference run is requested.
model = None
tokenizer = None

# In the submitted notebook this is usually skipped, because saved CSV outputs are reused.
if RUN_QWEN_MODEL:
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU available. Switch to a GPU runtime before running Qwen2.")

    # Clear GPU memory before loading the model to reduce Colab memory errors.
    gc.collect()
    torch.cuda.empty_cache()

    print("Loading model:", MODEL_NAME)

    # Load the tokenizer and model from Hugging Face and switch the model to evaluation mode.
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype="auto",
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    print("Model loaded.")
else:
    print("Qwen2 model loading skipped. Existing CSV files will be used.")

Qwen2 model loading skipped. Existing CSV files will be used.


## 7. Classification functions

The following functions construct Qwen2 prompts, generate model responses, parse JSON output, and save progress during inference.

During the original full run, results were written in batches. This was necessary because long Colab sessions disconnected. Batch saving made the classification process resumable.



### Build the Qwen2 prompt

This cell defines the prompt structure used for article classification. Each prompt contains the codebook, optional few-shot examples, and the article-specific classification text.

The model is asked to return a JSON object so that the response can be converted into stable dataframe columns. I used ChatGPT as coding support to implement the prompt structure and message formatting.



In [ ]:
# Build the fixed instruction that is reused for all article classifications.
def build_system_prompt():
    return f"""
Apply a deductive news value codebook to Dutch NU.nl article excerpts.

Classify each article at article level using six independent scores: 0, 50, or 100.

{PROMPT_CODEBOOK}

Scores must be assigned independently for each news value. Celebrity and good news should not be treated as exceptional categories. Code celebrity when an already famous person contributes to the article's newsworthiness. Code good news when positive outcomes such as wins, recoveries, rescues, breakthroughs, celebrations, or successful interventions are present, including when negative elements are also present.

Return only valid JSON with these fields:
- entertainment_score
- bad_news_score
- magnitude_score
- good_news_score
- celebrity_score
- power_elite_score
- dominant_news_values
- short_reason

Use this structure:
{{
  "entertainment_score": 0,
  "bad_news_score": 100,
  "magnitude_score": 50,
  "good_news_score": 0,
  "celebrity_score": 0,
  "power_elite_score": 50,
  "dominant_news_values": ["bad_news"],
  "short_reason": "Negative consequences are central; institutional actors are secondary."
}}
""".strip()


SYSTEM_PROMPT = build_system_prompt()


# Add the article-specific text and expected JSON format.
def build_user_prompt(classification_text, retry_instruction=""):
    retry_block = ""
    if retry_instruction:
        retry_block = f"""

Correction instruction:
The previous answer was rejected because: {retry_instruction}
Return a corrected JSON object only.
"""

    return f"""
Classify this NU.nl article excerpt according to the codebook.

ARTICLE TO CLASSIFY:
{classification_text}
{retry_block}

Return only the JSON object.
""".strip()


def shorten_for_fewshot(text, max_chars=FEWSHOT_EXAMPLE_CHAR_LIMIT):
    text = clean_for_prompt(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ... [excerpt shortened]"


# Convert manually checked examples into assistant-style JSON responses for few-shot prompting.
def manual_example_to_assistant_json(row):
    scores = {col: coerce_score(row.get(col, 0)) for col in NEWS_VALUES}
    obj = {
        **scores,
        "dominant_news_values": score_dict_to_dominant_values(scores),
        "short_reason": clean_for_prompt(row.get("manual_note", "Manual calibration example"))[:120] or "Manual calibration example",
    }
    return json.dumps(obj, ensure_ascii=False)


# Combine system instruction, optional few-shot examples, and article prompt into the message list.
def build_messages(classification_text, retry_instruction=""):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Add the completed few-shot examples before the article being classified.
    for _, example in fewshot_examples_for_prompt.iterrows():
        messages.append({"role": "user", "content": build_user_prompt(shorten_for_fewshot(example["classification_text"]))})
        messages.append({"role": "assistant", "content": manual_example_to_assistant_json(example)})

    messages.append({"role": "user", "content": build_user_prompt(classification_text, retry_instruction)})
    return messages

### Extract and validate JSON output

Qwen2 can return extra text around the requested JSON object or use unexpected score formats. These functions extract the JSON object, validate the required fields, and normalise scores to the allowed values.

I used ChatGPT as coding support for the parsing and validation implementation because this part of the workflow is mainly defensive programming rather than a substantive methodological choice.

In [ ]:
# Extract a valid JSON object from the raw Qwen2 response when possible.
def extract_json_object(text):
    if not isinstance(text, str):
        raise ValueError("Model output is not text.")

    cleaned = text.strip()
    # Remove Markdown code fences if they appear around the JSON.
    cleaned = re.sub(r"^```(?:json)?", "", cleaned, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()

    decoder = json.JSONDecoder()
    for start in [match.start() for match in re.finditer(r"\{", cleaned)]:
        try:
            parsed, _ = decoder.raw_decode(cleaned[start:])
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            return parsed

    raise ValueError(f"No valid JSON object found in output: {text[:300]}")


# Validate that all six required score fields are present.
def validate_news_value_output(parsed):
    missing = [col for col in NEWS_VALUES if col not in parsed]
    if missing:
        return False, f"Missing score fields: {missing}"

    # Normalise all model scores to the permitted values: 0, 50, or 100.
    scores = {col: coerce_score(parsed.get(col, 0)) for col in NEWS_VALUES}
    dominant = parsed.get("dominant_news_values", [])
    if isinstance(dominant, str):
        dominant = [dominant]
    if not isinstance(dominant, list) or not dominant:
        return False, "dominant_news_values must be a non-empty list."

    labels = [str(value).strip() for value in dominant]
    allowed_labels = {"entertainment", "bad_news", "magnitude", "good_news", "celebrity", "power_elite", "none"}
    invalid_labels = [label for label in labels if label not in allowed_labels]
    if invalid_labels:
        return False, f"Invalid dominant_news_values: {invalid_labels}"

    all_zero = all(score == 0 for score in scores.values())
    if all_zero and labels != ["none"]:
        return False, "dominant_news_values must be ['none'] when all scores are 0."
    if not all_zero and labels == ["none"]:
        return False, "dominant_news_values cannot be ['none'] when at least one score is above 0."

    return True, ""


# Convert parsed JSON into the exact output columns used in the CSV files.
def normalize_classification(parsed):
    row = {col: coerce_score(parsed.get(col, 0)) for col in NEWS_VALUES}
    dominant_values = score_dict_to_dominant_values(row)
    row["dominant_news_values"] = dominant_values_to_string(dominant_values)
    row["dominant_news_values_json"] = json.dumps(dominant_values, ensure_ascii=False)
    row["short_reason"] = clean_for_prompt(parsed.get("short_reason", ""))[:300]
    return row

### Classify one article

This cell defines the article-level classification function. It sends one prompt to Qwen2, parses the response, and retries when the output cannot be converted into the required JSON format.

If all retries fail, the article is saved with empty scores and `parse_success = False`. This keeps failed classifications visible in the output. I used ChatGPT as coding support for the retry and fallback logic.



In [ ]:
# Generate one raw Qwen2 response for one article prompt.
def generate_qwen_response(classification_text, max_new_tokens=140, retry_instruction=""):
    if model is None or tokenizer is None:
        raise RuntimeError("The model is not loaded. Set RUN_QWEN_MODEL to True if you want to run Qwen2.")

    # Convert prompt messages into model input tokens.
    messages = build_messages(classification_text, retry_instruction=retry_instruction)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
    input_device = next(model.parameters()).device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}

    # Generate without gradient tracking to reduce memory use during inference.
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Retry when JSON parsing or validation fails.
def classify_one_article(row):
    raw_output = ""
    last_error = ""
    normalized = None

    # Retry because language models sometimes return malformed output.
    for attempt in range(MAX_CLASSIFICATION_RETRIES + 1):
        try:
            raw_output = generate_qwen_response(
                row["classification_text"],
                retry_instruction=last_error if attempt > 0 else "",
            )
            parsed = extract_json_object(raw_output)
            is_valid, validation_error = validate_news_value_output(parsed)
            if not is_valid:
                last_error = validation_error
                continue

            normalized = normalize_classification(parsed)
            normalized["parse_ok"] = True
            normalized["parse_error"] = ""
            break
        except Exception as exc:
            last_error = repr(exc)

    # Save a fallback row after repeated failures so failed cases remain visible.
    if normalized is None:
        normalized = {col: 0 for col in NEWS_VALUES}
        normalized.update({
            "dominant_news_values": "none",
            "dominant_news_values_json": json.dumps(["none"], ensure_ascii=False),
            "short_reason": "PARSE_OR_VALIDATION_ERROR",
            "parse_ok": False,
            "parse_error": last_error,
        })

    normalized.update({
        "article_id": str(row["article_id"]),
        "title": clean_for_prompt(row.get(TITLE_COL, "")),
        "date": clean_for_prompt(row.get(DATE_COL, "")),
        "subjects": clean_for_prompt(row.get(SUBJECTS_COL, "")),
        "classification_text_char_count": int(row.get("classification_text_char_count", 0)),
        "body_char_limit": -1 if BODY_CHAR_LIMIT is None else BODY_CHAR_LIMIT,
        "model_name": MODEL_NAME,
        "prompt_type": "few-shot" if len(fewshot_examples_for_prompt) > 0 else "zero-shot_fallback",
        "fewshot_examples_used": int(len(fewshot_examples_for_prompt)),
    })
    return normalized, raw_output

### Run classification and save progress

This cell defines the batch-classification function used for long inference runs. It saves results at regular intervals and can resume from an existing output file by skipping article IDs that have already been classified. This design makes the workflow less fragile in Colab.


In [ ]:
# Append a batch of classification rows to the output CSV.
def append_csv_rows(rows, path):
    # Nothing is saved when the batch is empty.
    if not rows:
        return
    df = pd.DataFrame(rows)
    # Append to the existing file; write the header only if the file is new.
    df.to_csv(path, mode="a", header=not path.exists(), index=False, encoding="utf-8-sig")


# When resuming, skip article IDs that are already present in the saved output.
# Run classification for a dataframe of articles.
def run_classification(input_df, output_path, raw_output_path=None, resume=True):
    input_df = input_df.copy()
    input_df["article_id"] = input_df["article_id"].astype(str)

    # Track article IDs that have already been classified when resuming a run.
    done_ids = set()
    if resume and output_path.exists():
        existing_ids = pd.read_csv(output_path, usecols=["article_id"])
        done_ids = set(existing_ids["article_id"].astype(str))
        print(len(done_ids), "articles already saved in", output_path.name)

    todo = input_df[~input_df["article_id"].isin(done_ids)].copy()
    print("Articles to classify:", len(todo))

    # Store new rows temporarily and write them in batches to avoid losing progress.
    result_buffer = []
    raw_buffer = []
    start_time = time.time()

    for _, row in tqdm(todo.iterrows(), total=len(todo)):
        result, raw_output = classify_one_article(row)
        result_buffer.append(result)

        if raw_output_path is not None:
            raw_buffer.append({
                "article_id": str(row["article_id"]),
                "model_name": MODEL_NAME,
                "body_char_limit": -1 if BODY_CHAR_LIMIT is None else BODY_CHAR_LIMIT,
                "prompt_type": result.get("prompt_type", ""),
                "fewshot_examples_used": result.get("fewshot_examples_used", 0),
                "raw_output": raw_output,
                "parse_ok": result.get("parse_ok", False),
                "parse_error": result.get("parse_error", ""),
            })

        # Save progress regularly so a Colab disconnect does not lose everything.
        if len(result_buffer) >= SAVE_EVERY_N_ARTICLES:
            append_csv_rows(result_buffer, output_path)
            append_csv_rows(raw_buffer, raw_output_path) if raw_output_path is not None else None
            result_buffer.clear()
            raw_buffer.clear()

    append_csv_rows(result_buffer, output_path)
    append_csv_rows(raw_buffer, raw_output_path) if raw_output_path is not None else None

    elapsed_minutes = (time.time() - start_time) / 60
    print(f"Classification completed in {elapsed_minutes:.1f} minutes. Output: {output_path}")

## 8. Benchmark prevalence calibration

The raw Qwen2 scores are saved before any post-processing. I then create a separate benchmark-calibrated version of the scores for the main GIS workflow.

The calibration aligns the prevalence of each news value with the distribution reported by Harcup and O’Neill. It does not replace the raw Qwen2 output. The raw scores are preserved in separate columns and are later compared with the calibrated scores during manual validation.

I used ChatGPT as coding support for the implementation of the calibration procedure.



### Calibration method

The raw Qwen2 classifications and the benchmark-calibrated scores represent two different operationalisations of article-level news values.

For each news value, the notebook uses the Harcup and O’Neill benchmark distribution to calculate a target number of articles in this corpus. Articles are ranked using three components:

1. the original Qwen2 score;
2. keyword evidence from the article context;
3. a fixed article-ID tie-breaker to make the selection reproducible.

The highest-ranked articles are selected until the target count for each news value is reached. Raw Qwen2 scores are retained in `raw_*` columns, while the calibrated scores are saved in the main score columns. This makes it possible to evaluate later whether calibration improves, worsens, or leaves unchanged the agreement with manual scoring.

I used ChatGPT as coding support for the implementation of this ranking and export logic.

In [ ]:
# The procedure preserves raw Qwen2 scores and writes benchmark-calibrated scores separately.
# Keyword evidence is used only as a secondary ranking feature; the raw Qwen2 score remains the main ranking signal.

CALIBRATION_KEYWORDS = {
    "entertainment_score": [
        "show", "film", "serie", "muziek", "festival", "concert", "wedstrijd", "voetbal",
        "sport", "dier", "dieren", "humor", "grappig", "opmerkelijk", "lifestyle",
        "cultuur", "theater", "museum", "televisie", "media",
    ],
    "bad_news_score": [
        "dood", "overleden", "gewond", "ongeluk", "ongeval", "brand", "crisis",
        "conflict", "oorlog", "aanslag", "verdachte", "politie", "rechtbank",
        "celstraf", "ziek", "besmet", "schade", "dreiging", "verlies", "failliet",
    ],
    "magnitude_score": [
        "duizend", "duizenden", "miljoen", "miljoenen", "miljard", "miljarden",
        "record", "grootste", "landelijk", "nationaal", "massaal", "veel mensen",
        "regio", "provincie", "gemeenten", "impact",
    ],
    "good_news_score": [
        "wint", "gewonnen", "zege", "kampioen", "gered", "redding", "hersteld",
        "genezen", "doorbraak", "opgelost", "vrijgesproken", "feest", "viering",
        "succes", "teruggevonden", "opening", "verbetering", "akkoord", "herstel",
    ],
    "celebrity_score": [
        "koning", "koningin", "prins", "prinses", "máxima", "amalia", "willem-alexander",
        "artiest", "acteur", "actrice", "zanger", "zangeres", "presentator",
        "influencer", "rapper", "voetballer", "trainer", "coach", "bn'er", "beroemd",
    ],
    "power_elite_score": [
        "minister", "kabinet", "tweede kamer", "eerste kamer", "gemeente", "burgemeester",
        "politie", "rechtbank", "rechter", "openbaar ministerie", "om ", "ggd", "rivm",
        "rijksoverheid", "europese unie", "navo", "universiteit", "ziekenhuis", "bedrijf",
    ],
}


# Return an empty text column if a context column is missing.
def text_series(df, column):
    if column in df.columns:
        return df[column].fillna("").astype(str)
    return pd.Series("", index=df.index, dtype="object")


# Count how many calibration keywords appear in a piece of text.
def keyword_evidence_score(text, keywords):
    text = clean_for_prompt(text).lower()
    return sum(1 for keyword in keywords if keyword in text)


# Convert a benchmark percentage into a target number of articles for this dataset.
def target_count_for_sample(target_share, n_articles, minimum_one_if_positive=False):
    target_n = int(round(target_share * n_articles))
    if minimum_one_if_positive and target_share > 0 and n_articles > 0:
        target_n = max(1, target_n)
    return min(n_articles, target_n)


# Add article title, subjects, and classification text before calibration.
def add_article_context_for_calibration(results_df):
    df = results_df.copy()
    df["article_id"] = df["article_id"].astype(str)

    context_cols = ["article_id", TITLE_COL, DATE_COL, SUBJECTS_COL, "classification_text"]
    context_cols = [col for col in context_cols if col in articles_to_classify.columns]
    context_df = articles_to_classify[context_cols].copy()
    context_df["article_id"] = context_df["article_id"].astype(str)

    missing_context_cols = [col for col in context_cols if col != "article_id" and col not in df.columns]
    if missing_context_cols:
        df = df.merge(context_df[["article_id", *missing_context_cols]], on="article_id", how="left")

    return df


# Read the six news value scores from one row as a dictionary.
def score_dict_from_row(row):
    return {col: coerce_score(row.get(col, 0)) for col in NEWS_VALUES}

# Rank articles for each news value and select the benchmark target count reproducibly.
def calibrate_to_harcup_oneill_distribution(results_df, target_shares, minimum_one_if_positive=False):
    # This function makes a calibrated version of the model scores.
    # For each news value, it selects approximately the number of articles that would match the Harcup and O'Neill benchmark distribution.
    # I keep the raw scores in separate columns so I can still inspect them later.
    df = add_article_context_for_calibration(results_df)
    n_articles = len(df)
    if n_articles == 0:
        raise ValueError("Cannot calibrate an empty dataframe.")

    # Combine article context into one text field for keyword matching.
    calibration_text = (
        text_series(df, "classification_text") + " "
        + text_series(df, TITLE_COL) + " "
        + text_series(df, SUBJECTS_COL) + " "
        + text_series(df, "short_reason")
    )

    calibration_rows = []
    for col in NEWS_VALUES:
        if col not in df.columns:
            df[col] = 0

        # Keep the original Qwen2 score separately before replacing the calibrated score.
        raw_col = f"raw_{col}"
        rank_col = f"{col}_calibration_rank_score"
        df[raw_col] = df[col].apply(coerce_score)

        # Keyword evidence is added to the ranking, but the raw Qwen2 score remains most important.
        keyword_scores = calibration_text.apply(
            lambda text: keyword_evidence_score(text, CALIBRATION_KEYWORDS.get(col, []))
        )
        df[rank_col] = (df[raw_col] * 100) + keyword_scores

        # Calculate the target number of positive articles for this news value.
        target_share = target_shares[col]
        target_n = target_count_for_sample(
            target_share,
            n_articles,
            minimum_one_if_positive=minimum_one_if_positive,
        )

        # Use a fixed article-ID hash to break ties reproducibly.
        tiebreak = pd.util.hash_pandas_object(df["article_id"].astype(str), index=False).astype("uint64")
        ranked = pd.DataFrame({
            "rank_score": df[rank_col],
            "raw_score": df[raw_col],
            "tiebreak": tiebreak,
        }, index=df.index).sort_values(
            by=["rank_score", "raw_score", "tiebreak"],
            ascending=[False, False, True],
        )

        # Select the highest-ranked articles and assign calibrated scores.
        selected_index = ranked.head(target_n).index
        df[col] = 0
        selected_raw_scores = df.loc[selected_index, raw_col].apply(coerce_score)
        df.loc[selected_index, col] = selected_raw_scores.where(selected_raw_scores > 0, 50)

        # Store summary information so the calibration can be checked later.
        calibration_rows.append({
            "news_value": col.replace("_score", ""),
            "benchmark_count": HARCUP_ONEILL_TARGET_COUNTS[col],
            "benchmark_n": HARCUP_ONEILL_REFERENCE_N,
            "target_share": target_share,
            "target_n_for_this_sample": target_n,
            "raw_present_n": int((df[raw_col] > 0).sum()),
            "calibrated_present_n": int((df[col] > 0).sum()),
            "calibrated_present_share": float((df[col] > 0).mean()),
            "selected_from_raw_zero_n": int((df.loc[selected_index, raw_col] == 0).sum()),
        })

    # Recalculate dominant news values after calibration.
    df["dominant_news_values"] = df.apply(
        lambda row: dominant_values_to_string(score_dict_to_dominant_values(score_dict_from_row(row))),
        axis=1,
    )
    df["dominant_news_values_json"] = df.apply(
        lambda row: json.dumps(score_dict_to_dominant_values(score_dict_from_row(row)), ensure_ascii=False),
        axis=1,
    )

    # Remove temporary helper columns before saving the final calibrated file.
    helper_cols = [col for col in df.columns if col.endswith("_calibration_rank_score")]
    helper_cols += ["classification_text"]
    calibrated_df = df.drop(columns=helper_cols, errors="ignore")
    calibration_summary = pd.DataFrame(calibration_rows)
    return calibrated_df, calibration_summary


# Save the calibrated article-level output and a summary table documenting the calibration effect.
def save_calibrated_output(input_df, calibrated_path, summary_path, minimum_one_if_positive=False):
    calibrated_df, calibration_summary = calibrate_to_harcup_oneill_distribution(
        input_df,
        HARCUP_ONEILL_TARGET_SHARES,
        minimum_one_if_positive=minimum_one_if_positive,
    )
    calibrated_df.to_csv(calibrated_path, index=False, encoding="utf-8-sig")
    calibration_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
    print("Saved calibrated results:", calibrated_path)
    print("Saved calibration summary:", summary_path)
    return calibrated_df, calibration_summary

## 9. Smoke test

The smoke test classifies a small number of articles to check whether the prompt, model call, and JSON parser work together.

This step is only needed when changing the prompt or rerunning Qwen2. It is skipped in the submitted workflow because the full classification output already exists.


In [ ]:
# A smoke test classifies only a few articles to check that the model workflow works.
TEST_N = 5

# Run a small classification test only when Qwen2 has been loaded.
if RUN_QWEN_MODEL:
    test_rows = []
    raw_rows = []

    # Keep both parsed rows and raw model text so parsing failures can be inspected.
    for _, row in articles_to_classify.head(TEST_N).iterrows():
        result, raw_output = classify_one_article(row)
        test_rows.append(result)
        raw_rows.append({"article_id": row["article_id"], "raw_output": raw_output})

    smoke_test_df = pd.DataFrame(test_rows)
    display(smoke_test_df)

    for raw_row in raw_rows:
        print("\n---", raw_row["article_id"], "---")
        print(raw_row["raw_output"][:1000])

else:
    print("Smoke test skipped because Qwen2 is not loaded.")

Smoke test skipped because Qwen2 is not loaded.


## 10. Pilot classification

The pilot classification was used to inspect model behaviour before running the full corpus. If the pilot file already exists, the notebook loads it instead of regenerating it.

The pilot is not rerun in the submitted workflow because the final full classification has already been completed.


In [ ]:
# Reuse the saved pilot output when available.
if PILOT_PATH.exists():
    print("Pilot file already exists:", PILOT_PATH)


elif RUN_QWEN_MODEL:
    fewshot_ids = set()
    # Exclude completed few-shot examples from the pilot pool where possible.
    if not completed_fewshot_examples.empty:
        fewshot_ids = set(completed_fewshot_examples["article_id"].astype(str))

    # Create the pool of articles that can be sampled for the pilot.
    pilot_pool = articles_to_classify[~articles_to_classify["article_id"].astype(str).isin(fewshot_ids)].copy()

    if pilot_pool.empty:
        raise ValueError("Pilot pool is empty after excluding few-shot examples.")

    # Draw a reproducible pilot sample when the pilot has to be created.
    pilot_df = pilot_pool.sample(
        n=min(PILOT_N, len(pilot_pool)),
        random_state=RANDOM_SEED + 1,
    ).copy()

    run_classification(pilot_df, PILOT_PATH, raw_output_path=None, resume=False)

else:
    print("Pilot classification skipped. Existing pilot file was not found.")

if PILOT_PATH.exists():
    # Display basic pilot checks after loading or creation.
    pilot_results = pd.read_csv(PILOT_PATH)
    print("Pilot rows:", len(pilot_results))

    if "parse_ok" in pilot_results.columns:
        print("Parse success rate:", pilot_results["parse_ok"].mean())

    display(pilot_results.head(20))

Pilot file already exists: /content/drive/MyDrive/Thesis/data_processed/pilot_article_news_values.csv
Pilot rows: 150
Parse success rate: 0.98


,entertainment_score,bad_news_score,magnitude_score,good_news_score,celebrity_score,power_elite_score,dominant_news_values,dominant_news_values_json,short_reason,parse_ok,parse_error,article_id,title,date,subjects,classification_text_char_count,body_char_limit,model_name,prompt_type,fewshot_examples_used
0,100,0,50,100,0,0,entertainment; good_news,"[""entertainment"", ""good_news""]",Tennis player reaches final after overcoming m...,True,NaN,11241,Elise Mertens overleeft maar liefst elf matchp...,14 juni 2025 zaterdag 12:15 PM GMT,Sports + Recreation Events; Tournaments; Tenni...,1043,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
1,0,0,0,100,0,100,good_news; power_elite,"[""good_news"", ""power_elite""]",Sports coach interviews his ideal team; positi...,True,NaN,40848,Reiziger lijkt Jong Oranje-puzzel te hebben ge...,20 juni 2025 vrijdag 06:24 PM GMT,Sports + Recreation Events; Tournaments; Socce...,1073,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
2,0,0,0,0,0,0,none,"[""none""]",No news values present.,True,NaN,1944,Ajax blijft steken op gelijkspel bij tiental N...,20 december 2025 zaterdag 08:57 PM GMT,Soccer; Sports & Recreation Events,1031,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
3,100,0,0,0,0,0,entertainment,"[""entertainment""]",Baritonsaxophonist brings queer stories and cl...,True,NaN,4055,Baritonsaxofonist brengt queer verhalen en kla...,20 juni 2025 vrijdag 09:30 AM GMT,Music; Colleges + Universities; Music Groups +...,1140,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
4,0,0,0,0,0,0,none,"[""none""]",No news values present.,True,NaN,43903,Steeds meer 25-plussers in Utrecht blijven thu...,2 oktober 2025 donderdag 10:18 AM GMT,Parents; Adolescents & Teens; Housing Market; ...,1109,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
5,0,100,50,0,0,0,bad_news,"[""bad_news""]",Heat-related illnesses reported at events in L...,True,NaN,31987,Meerdere hardlopers onwel bij evenementen in L...,11 mei 2025 zondag 01:17 PM GMT,NaN,485,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
6,0,0,0,0,0,0,none,"[""none""]",NaN,True,NaN,8759,Deze prins komt van boven de sloot en houdt ni...,19 februari 2025 woensdag 08:12 AM GMT,Holidays + Observances; Communities + Neighbor...,1122,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
7,100,0,0,0,100,0,entertainment; celebrity,"[""entertainment"", ""celebrity""]",Celebrity returns to Netherlands; sports event...,True,NaN,49109,Van Dijk met kids het veld op na goal in Groni...,11 juni 2025 woensdag 02:30 AM GMT,"Soccer; Sports + Recreation Events; Children, ...",1055,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
8,0,100,50,0,0,50,bad_news,"[""bad_news""]",High-profile drug-smuggling case; high-level c...,True,NaN,32706,Miljoenenwinst met cocaïne: Rachid el H. opgepakt,20 augustus 2025 woensdag 09:14 AM GMT,Police Forces; Illegal Drugs; Law Courts + Tri...,1277,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
9,0,0,0,0,0,0,none,"[""none""]",No news values present.,True,NaN,20365,Koeman hint op wijzigingen bij Oranje en speel...,16 november 2025 zondag 02:27 PM GMT,Sports & Recreation Events; Press Conferences;...,1075,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4


In [ ]:
# Inspect pilot score distributions and parsing failures before relying on the full run.
if PILOT_PATH.exists():
    pilot_results = pd.read_csv(PILOT_PATH)

    # Print the score distribution for each separate news value.
    for col in NEWS_VALUES:
        print("\n", col)
        print(pilot_results[col].value_counts(dropna=False).sort_index())

    print("\nDominant news values distribution:")
    print(pilot_results["dominant_news_values"].value_counts(dropna=False))

    # Show problematic rows if the JSON parser failed for any pilot articles.
    if "parse_ok" in pilot_results.columns:
        display(pilot_results[pilot_results["parse_ok"] == False].head(20))
else:
    print("Pilot file not found.")


 entertainment_score
entertainment_score
0      137
50       1
100     12
Name: count, dtype: int64

 bad_news_score
bad_news_score
0      109
100     41
Name: count, dtype: int64

 magnitude_score
magnitude_score
0      104
50      42
100      4
Name: count, dtype: int64

 good_news_score
good_news_score
0      132
100     18
Name: count, dtype: int64

 celebrity_score
celebrity_score
0      147
100      3
Name: count, dtype: int64

 power_elite_score
power_elite_score
0      109
50      35
100      6
Name: count, dtype: int64

Dominant news values distribution:
dominant_news_values
none                                   74
bad_news                               41
good_news                              14
entertainment                           7
power_elite                             4
magnitude                               3
entertainment; good_news                2
entertainment; celebrity                2
good_news; power_elite                  1
magnitude; power_elite        

,entertainment_score,bad_news_score,magnitude_score,good_news_score,celebrity_score,power_elite_score,dominant_news_values,dominant_news_values_json,short_reason,parse_ok,parse_error,article_id,title,date,subjects,classification_text_char_count,body_char_limit,model_name,prompt_type,fewshot_examples_used
74,0,0,0,0,0,0,none,"[""none""]",PARSE_OR_VALIDATION_ERROR,False,dominant_news_values must be a non-empty list.,28902,Live | Stoet van anti-immigratiebetoging trekt...,12 oktober 2025 zondag 10:19 AM GMT,NaN,519,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
81,0,0,0,0,0,0,none,"[""none""]",PARSE_OR_VALIDATION_ERROR,False,dominant_news_values must be a non-empty list.,25598,Live F1 | Regen in aanloop naar tweede vrije t...,29 augustus 2025 vrijdag 08:06 AM GMT,Auto Racing; Tournaments; Auto Racing; Tournam...,673,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
85,0,0,0,0,0,0,none,"[""none""]",PARSE_OR_VALIDATION_ERROR,False,dominant_news_values must be a non-empty list.,21629,Lieke Klaver dacht aan Femke Bol tijdens goude...,8 maart 2025 zaterdag 10:18 PM GMT,Tournaments; Sports + Recreation Events; Women,1035,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4


## 11. Full classification

This is the full Qwen2 inference step that produced `article_news_values.csv` and `article_news_values_raw_outputs.csv`.

The submitted notebook does not rerun this step. It loads the saved output created during the original classification run. Reproducing the full inference run from scratch requires `RUN_QWEN_MODEL = True`, the same model configuration, GPU availability, and the completed few-shot examples used in the original run.



In [ ]:
# Run full inference only when deliberately replacing or reproducing the saved Qwen2 output.
if RUN_QWEN_MODEL:
    run_classification(
        articles_to_classify,
        NEWS_VALUES_PATH,
        raw_output_path=RAW_OUTPUTS_PATH,
        resume=True,
    )
else:
    print("Full classification skipped. Existing output will be used if available.")

Full classification skipped. Existing output will be used if available.


## 12. Check full classification output

This step loads the saved full classification file and checks row counts, parsing success, prompt type, and score distributions.


In [ ]:
# Load the saved full classification output and inspect reproducibility checks.
if NEWS_VALUES_PATH.exists():
    news_values_df = pd.read_csv(NEWS_VALUES_PATH)
    news_values_df["article_id"] = news_values_df["article_id"].astype(str)

    print("Classified rows:", len(news_values_df))
    print("Unique classified articles:", news_values_df["article_id"].nunique())
    if "parse_ok" in news_values_df.columns:
        print("Parse success rate:", news_values_df["parse_ok"].mean())
    if "prompt_type" in news_values_df.columns:
        print("Prompt types:")
        print(news_values_df["prompt_type"].value_counts(dropna=False))

    display(news_values_df.head())

    # Report score distributions so unexpected patterns are visible before calibration.
    for col in NEWS_VALUES:
        print("\n", col)
        print(news_values_df[col].value_counts(dropna=False).sort_index())

    print("\nDominant news values distribution:")
    print(news_values_df["dominant_news_values"].value_counts(dropna=False).head(30))
else:
    print("No full classification file found:", NEWS_VALUES_PATH)

Classified rows: 12225
Unique classified articles: 12225
Parse success rate: 0.9879754601226994
Prompt types:
prompt_type
few-shot    12225
Name: count, dtype: int64


,entertainment_score,bad_news_score,magnitude_score,good_news_score,celebrity_score,power_elite_score,dominant_news_values,dominant_news_values_json,short_reason,parse_ok,parse_error,article_id,title,date,subjects,classification_text_char_count,body_char_limit,model_name,prompt_type,fewshot_examples_used
0,0,100,50,0,0,50,bad_news,"[""bad_news""]",Health alert about hepatitis A virus infection...,True,NaN,436,12 mensen (erg) ziek: dit weten we over het he...,14 januari 2025 dinsdag 09:52 AM GMT,Government Departments + Authorities; Hepatiti...,1211,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
1,0,100,50,0,0,50,bad_news,"[""bad_news""]",Infectious disease outbreak; high magnitude; l...,True,NaN,438,"12 zieken, 2 opnames: dit weten we over het he...",14 januari 2025 dinsdag 09:52 AM GMT,Hepatitis; Infectious Disease; Viruses; Govern...,1211,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
2,0,100,50,0,0,50,bad_news,"[""bad_news""]",Fraudulent webshop exposed by police; high-pro...,True,NaN,443,125.000 mensen bezochten nepwebshop waarmee po...,25 januari 2025 zaterdag 09:06 AM GMT,Fraud + Financial Crime; Mail Order + Internet...,1149,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
3,0,0,100,0,0,0,magnitude,"[""magnitude""]",Fraudulent website targeted by police; high-pr...,True,NaN,444,125.000 mensen bezoeken nepwebshop waarmee pol...,25 januari 2025 zaterdag 09:06 AM GMT,Mail Order + Internet Retailing; Fraud + Finan...,1148,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4
4,0,100,50,0,0,50,bad_news,"[""bad_news""]",Police raids on multiple explosive incidents; ...,True,NaN,468,15 arrestaties voor meerdere explosies in Rott...,20 januari 2025 maandag 03:02 PM GMT,Arrests; Police Forces; Misdemeanors; Criminal...,1127,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4



 entertainment_score
entertainment_score
0      10680
50        12
100     1533
Name: count, dtype: int64

 bad_news_score
bad_news_score
0      8672
100    3553
Name: count, dtype: int64

 magnitude_score
magnitude_score
0      7756
50     3782
100     687
Name: count, dtype: int64

 good_news_score
good_news_score
0      10726
50        55
100     1444
Name: count, dtype: int64

 celebrity_score
celebrity_score
0      11799
50       150
100      276
Name: count, dtype: int64

 power_elite_score
power_elite_score
0      8729
50     3021
100     475
Name: count, dtype: int64

Dominant news values distribution:
dominant_news_values
none                                                5323
bad_news                                            3538
entertainment                                        954
good_news                                            766
magnitude                                            457
entertainment; good_news                             333
power_elite       

## 13. Create or load benchmark-calibrated scores

This step creates a benchmark-calibrated version of the Qwen2 scores when it does not already exist. The raw Qwen2 output remains saved separately. Because calibration changes score distributions, it is documented as a separate post-processing step and was implemented with ChatGPT coding support.


### Apply calibration to the full classification file

The Harcup and O’Neill benchmark counts are converted into target shares and then applied to the article sample used in this thesis.

If the calibrated file already exists, the notebook loads it instead of recalculating it. If it does not exist, the notebook creates it from the raw Qwen2 classification output. The calibrated output is saved separately from the raw output.



In [ ]:
# Convert Harcup and O’Neill benchmark counts into target shares for this corpus.
HARCUP_ONEILL_REFERENCE_N = 711

HARCUP_ONEILL_TARGET_COUNTS = {
    "bad_news_score": 442,
    "entertainment_score": 332,
    "power_elite_score": 216,
    "magnitude_score": 165,
    "celebrity_score": 145,
    "good_news_score": 137,
}

HARCUP_ONEILL_TARGET_SHARES = {
    col: count / HARCUP_ONEILL_REFERENCE_N
    for col, count in HARCUP_ONEILL_TARGET_COUNTS.items()
}

# Reuse the saved calibrated file when available; otherwise create it from the raw Qwen2 output.
if CALIBRATED_NEWS_VALUES_PATH.exists():
    print("Calibrated file already exists:", CALIBRATED_NEWS_VALUES_PATH)
    calibrated_news_values_df = pd.read_csv(CALIBRATED_NEWS_VALUES_PATH)

    if CALIBRATION_SUMMARY_PATH.exists():
        calibration_summary = pd.read_csv(CALIBRATION_SUMMARY_PATH)
        display(calibration_summary)

    display(calibrated_news_values_df.head())

# Keep the raw classification file unchanged and save calibration diagnostics separately.
elif NEWS_VALUES_PATH.exists():
    news_values_df = pd.read_csv(NEWS_VALUES_PATH)
    calibrated_news_values_df, calibration_summary = save_calibrated_output(
        news_values_df,
        CALIBRATED_NEWS_VALUES_PATH,
        CALIBRATION_SUMMARY_PATH,
        minimum_one_if_positive=False,
    )
    display(calibration_summary)
    display(calibrated_news_values_df.head())

else:
    print("Full classification output not found. Run the full classification cell first.")

Calibrated file already exists: /content/drive/MyDrive/Thesis/data_processed/article_news_values_calibrated_to_harcup_oneill.csv


,news_value,benchmark_count,benchmark_n,target_share,target_n_for_this_sample,raw_present_n,calibrated_present_n,calibrated_present_share,selected_from_raw_zero_n
0,entertainment,332,711,0.466948,5708,1545,5708,0.466912,4163
1,bad_news,442,711,0.621660,7600,3553,7600,0.621677,4047
2,magnitude,165,711,0.232068,2837,4469,2837,0.232065,0
3,good_news,137,711,0.192686,2356,1499,2356,0.192720,857
4,celebrity,145,711,0.203938,2493,426,2493,0.203926,2067
5,power_elite,216,711,0.303797,3714,3496,3714,0.303804,218


,entertainment_score,bad_news_score,magnitude_score,good_news_score,celebrity_score,power_elite_score,dominant_news_values,dominant_news_values_json,short_reason,parse_ok,...,body_char_limit,model_name,prompt_type,fewshot_examples_used,raw_entertainment_score,raw_bad_news_score,raw_magnitude_score,raw_good_news_score,raw_celebrity_score,raw_power_elite_score
0,0,100,50,0,0,50,bad_news,"[""bad_news""]",Health alert about hepatitis A virus infection...,True,...,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4,0,100,50,0,0,50
1,0,100,0,0,0,50,bad_news,"[""bad_news""]",Infectious disease outbreak; high magnitude; l...,True,...,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4,0,100,50,0,0,50
2,0,100,50,0,0,50,bad_news,"[""bad_news""]",Fraudulent webshop exposed by police; high-pro...,True,...,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4,0,100,50,0,0,50
3,0,50,100,0,0,0,magnitude,"[""magnitude""]",Fraudulent website targeted by police; high-pr...,True,...,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4,0,0,100,0,0,0
4,0,100,50,0,0,50,bad_news,"[""bad_news""]",Police raids on multiple explosive incidents; ...,True,...,750,Qwen/Qwen2-1.5B-Instruct,few-shot,4,0,100,50,0,0,50


## 14. Manual validation sample

This section creates or loads the sample used for manual validation. Once the sample exists, the notebook loads the same file again so that the manually checked articles remain stable across reruns.

The validation sample is used to assess whether the Qwen2 labels are plausible at article level.

The validation sample is used in Notebook 3 to compare manual scores with both calibrated and raw Qwen2 scores.



In [ ]:
# Select the best available classification file for validation-sample creation.
def load_validation_base():
    if CALIBRATED_NEWS_VALUES_PATH.exists():
        return pd.read_csv(CALIBRATED_NEWS_VALUES_PATH), "full_calibrated"
    if NEWS_VALUES_PATH.exists():
        return pd.read_csv(NEWS_VALUES_PATH), "full"
    if PILOT_CALIBRATED_PATH.exists():
        return pd.read_csv(PILOT_CALIBRATED_PATH), "pilot_calibrated"
    if PILOT_PATH.exists():
        return pd.read_csv(PILOT_PATH), "pilot"
    raise FileNotFoundError("Neither the full classification file nor the pilot file exists.")


# Keep the same validation sample once it has been created.
if VALIDATION_SAMPLE_PATH.exists():
    validation_sample = pd.read_csv(VALIDATION_SAMPLE_PATH)
    print("Existing validation sample loaded:", VALIDATION_SAMPLE_PATH)
    print("Delete this file to regenerate the validation sample.")
else:
    # Use the best available classification output as the basis for manual validation.
    # Prefer the calibrated full output, because Notebook 3 compares calibrated and raw scores.
    validation_base, validation_base_source = load_validation_base()
    validation_base["article_id"] = validation_base["article_id"].astype(str)
    print("Validation base:", validation_base_source)

    # Exclude articles that were already used in few-shot examples or the pilot.
    exclude_ids = set()

    if not completed_fewshot_examples.empty:
        exclude_ids.update(completed_fewshot_examples["article_id"].astype(str))

    if PILOT_PATH.exists():
        pilot_ids = pd.read_csv(PILOT_PATH, usecols=["article_id"])["article_id"].astype(str)
        exclude_ids.update(pilot_ids)

    validation_base = validation_base[~validation_base["article_id"].isin(exclude_ids)].copy()

    if validation_base.empty:
        raise ValueError("Validation pool is empty after exclusions.")

    # Add article context so manual validation is easier to do.
    context_cols = ["article_id", TITLE_COL, DATE_COL, SUBJECTS_COL, "classification_text"]
    context_cols = [col for col in context_cols if col in articles_to_classify.columns]

    context_df = articles_to_classify[context_cols].copy()
    context_df["article_id"] = context_df["article_id"].astype(str)

    # Draw a reproducible random validation sample and merge article context onto it.
    validation_sample = validation_base.sample(
        n=min(VALIDATION_N, len(validation_base)),
        random_state=RANDOM_SEED + 2,
    ).merge(context_df, on="article_id", how="left")

    # Add empty columns where I can enter my manual validation scores.
    for col in NEWS_VALUES:
        validation_sample[f"manual_{col}"] = ""

    # Put the most useful columns first in the validation CSV.
    ordered_cols = ["article_id"]
    ordered_cols += [col for col in [TITLE_COL, DATE_COL, SUBJECTS_COL, "classification_text"] if col in validation_sample.columns]
    ordered_cols += NEWS_VALUES
    ordered_cols += [f"manual_{col}" for col in NEWS_VALUES]
    ordered_cols += [col for col in ["dominant_news_values", "short_reason", "prompt_type", "fewshot_examples_used"] if col in validation_sample.columns]

    other_cols = [col for col in validation_sample.columns if col not in ordered_cols]
    validation_sample = validation_sample[ordered_cols + other_cols]

    # Save the validation sample so the same manually coded articles are used in later reruns.
    validation_sample.to_csv(VALIDATION_SAMPLE_PATH, index=False, encoding="utf-8-sig")
    print("Validation sample saved:", VALIDATION_SAMPLE_PATH)

display(validation_sample.head())

Existing validation sample loaded: /content/drive/MyDrive/Thesis/data_processed/manual_validation_sample.csv
Delete this file to regenerate the validation sample.


,article_id,classification_text,entertainment_score,bad_news_score,magnitude_score,good_news_score,celebrity_score,power_elite_score,manual_entertainment_score,manual_bad_news_score,...,model_name,raw_entertainment_score,raw_bad_news_score,raw_magnitude_score,raw_good_news_score,raw_celebrity_score,raw_power_elite_score,title_y,date_y,subjects_y
0,36459,Article ID: 36459\nDate: 14 november 2025 vrij...,0,50,100,0,0,0,0,100,...,Qwen/Qwen2-1.5B-Instruct,0,0,100,0,0,0,Ook 'lichte' bevingen in Groningen leiden tot ...,14 november 2025 vrijdag 07:41 AM GMT,Earthquakes; Natural Disasters; Emergency Serv...
1,48187,Article ID: 48187\nDate: 1 januari 2025 woensd...,0,100,50,0,0,50,0,100,...,Qwen/Qwen2-1.5B-Instruct,0,100,50,0,0,50,Tweede dodelijk slachtoffer door vuurwerk: man...,1 januari 2025 woensdag 05:50 PM GMT,Death + Dying; Police Forces; Wounds + Injurie...
2,25926,Article ID: 25926\nDate: 28 augustus 2025 dond...,50,0,0,100,0,0,100,0,...,Qwen/Qwen2-1.5B-Instruct,0,0,0,100,0,0,Live F1 | Verstappen staat pers te woord in aa...,28 augustus 2025 donderdag 09:52 AM GMT,Auto Racing; Tournaments; Auto Racing; Tournam...
3,17122,Article ID: 17122\nDate: 21 januari 2025 dinsd...,0,50,0,0,0,0,0,50,...,Qwen/Qwen2-1.5B-Instruct,0,0,0,0,0,0,Hospita in Enschede voor economisch daklozen,21 januari 2025 dinsdag 10:31 AM GMT,Social Security; Housing Market; Poverty + Hom...
4,8291,Article ID: 8291\nDate: 20 maart 2025 donderda...,50,50,0,50,0,0,100,0,...,Qwen/Qwen2-1.5B-Instruct,0,0,0,0,0,0,De voorzitter zonder Elfstedentocht: 'Ook de t...,20 maart 2025 donderdag 08:40 PM GMT,Sports + Recreation Events; Ice Skating; Assoc...
